# Version 2 — Cyclone Prediction with ERA5 Weather Features
Extends Version 1 by adding ERA5 atmospheric and oceanic variables matched to each storm observation.

In [ ]:
!pip -q install pandas numpy scikit-learn xgboost matplotlib seaborn xarray netCDF4

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import math
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)
from xgboost import XGBClassifier

print('Imports OK')

## Step 1: Load IBTrACS

In [ ]:
IBTRACS_URL = (
    'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs'
    '/v04r01/access/csv/ibtracs.NI.list.v04r01.csv'
)

print('Loading IBTrACS …')
df_raw = pd.read_csv(IBTRACS_URL, skiprows=[1], low_memory=False)

for col in ['LAT', 'LON', 'USA_WIND', 'WMO_WIND', 'USA_PRES']:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df_raw['ISO_TIME'] = pd.to_datetime(df_raw['ISO_TIME'], utc=True, errors='coerce')

if 'USA_WIND' in df_raw.columns and 'WMO_WIND' in df_raw.columns:
    df_raw['WIND'] = df_raw['USA_WIND'].fillna(df_raw['WMO_WIND'])
elif 'USA_WIND' in df_raw.columns:
    df_raw['WIND'] = df_raw['USA_WIND']
elif 'WMO_WIND' in df_raw.columns:
    df_raw['WIND'] = df_raw['WMO_WIND']
else:
    df_raw['WIND'] = np.nan

df_raw['WIND'] = pd.to_numeric(df_raw['WIND'], errors='coerce')
df_raw = df_raw.dropna(subset=['SID', 'ISO_TIME', 'WIND'])
df_raw = df_raw.sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)

# Filter Bay of Bengal
df = df_raw[
    df_raw['LAT'].between(0, 25) & df_raw['LON'].between(80, 100)
].reset_index(drop=True)

print(f'Bay of Bengal rows: {len(df)}, unique storms: {df["SID"].nunique()}')

## Step 2: Upload ERA5 NetCDF File
Download ERA5 from https://cds.climate.copernicus.eu — select variables: sst, msl, u10, v10, t2m, d2m, tp.  
Region: 0–25N, 80–100E. Format: NetCDF.

In [ ]:
from google.colab import files

print('Upload your ERA5 NetCDF file (.nc):')
uploaded = files.upload()

era5_path = list(uploaded.keys())[0]
print(f'\nOpening: {era5_path}')
ds = xr.open_dataset(era5_path)
print('\nVariables:', list(ds.data_vars))
print('Dimensions:', dict(ds.dims))
print('Coordinates:', list(ds.coords))

In [ ]:
# ── Coordinate Normalisation ──────────────────────────────────────────────────
# ERA5 may use 'latitude'/'longitude' or 'lat'/'lon', and 'valid_time' or 'time'

rename_map = {}
coords = list(ds.coords)

if 'latitude' in coords and 'lat' not in coords:
    rename_map['latitude'] = 'lat'
if 'longitude' in coords and 'lon' not in coords:
    rename_map['longitude'] = 'lon'
if 'valid_time' in coords and 'time' not in coords:
    rename_map['valid_time'] = 'time'

if rename_map:
    ds = ds.rename(rename_map)
    print('Renamed coordinates:', rename_map)

print('\nFinal coordinate names:', list(ds.coords))
print('Lat range :', float(ds['lat'].min().values), '–', float(ds['lat'].max().values))
print('Lon range :', float(ds['lon'].min().values), '–', float(ds['lon'].max().values))
print('Time range:', str(ds['time'].values[0])[:19], '–', str(ds['time'].values[-1])[:19])

In [ ]:
# ── Match ERA5 Variables to Storm Tracks ─────────────────────────────────────

# Map our desired feature names to possible ERA5 variable names
VARIABLE_MAP = {
    'SST':    ['sst', 'sea_surface_temperature'],
    'MSL':    ['msl', 'mean_sea_level_pressure'],
    'U10':    ['u10', '10m_u_component_of_wind'],
    'V10':    ['v10', '10m_v_component_of_wind'],
    'T2M':    ['t2m', '2m_temperature'],
    'D2M':    ['d2m', '2m_dewpoint_temperature'],
    'TP':     ['tp',  'total_precipitation'],
}

available_vars = list(ds.data_vars)
resolved_vars = {}   # feat_name -> ds variable name
for feat_name, candidates in VARIABLE_MAP.items():
    for c in candidates:
        if c in available_vars:
            resolved_vars[feat_name] = c
            break

print('Resolved ERA5 variables:', resolved_vars)


def match_era5_to_tracks(df, ds, resolved_vars):
    """Extract ERA5 values at the nearest grid point and time for each track row."""
    result_df = df.copy()
    for feat_name in resolved_vars:
        result_df[feat_name] = np.nan

    for i, row in result_df.iterrows():
        try:
            lat_val = float(row['LAT'])
            lon_val = float(row['LON'])
            t_val   = row['ISO_TIME']

            # Convert tz-aware to tz-naive for xarray compatibility
            if hasattr(t_val, 'tz_localize'):
                t_val = t_val.tz_localize(None)
            elif hasattr(t_val, 'tz_convert'):
                t_val = t_val.tz_convert(None)
            t_np = np.datetime64(t_val)

            point = ds.sel(lat=lat_val, lon=lon_val, time=t_np, method='nearest')
            for feat_name, var_name in resolved_vars.items():
                try:
                    val = float(point[var_name].values)
                    result_df.at[i, feat_name] = val
                except Exception:
                    pass
        except Exception:
            pass

        if i % 500 == 0:
            print(f'  Matched {i}/{len(result_df)} rows…', end='\r')

    print(f'\nDone. ERA5 columns added: {list(resolved_vars.keys())}')
    return result_df


df = match_era5_to_tracks(df, ds, resolved_vars)
df[list(resolved_vars.keys())].describe()

In [ ]:
# ── Track Features & Labels ───────────────────────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2)**2
    return R * 2 * math.asin(math.sqrt(max(0.0, min(1.0, a))))


def add_track_features(df):
    df = df.copy().sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)
    df['MONTH'] = df['ISO_TIME'].dt.month.astype(float)
    df['HOUR']  = df['ISO_TIME'].dt.hour.astype(float)
    grp = df.groupby('SID', sort=False)
    df['LAT_CHANGE']  = grp['LAT'].diff()
    df['LON_CHANGE']  = grp['LON'].diff()
    df['WIND_CHANGE'] = grp['WIND'].diff()
    speeds = np.full(len(df), np.nan)
    for sid, group in df.groupby('SID', sort=False):
        idxs  = group.index.tolist()
        lats  = group['LAT'].values
        lons  = group['LON'].values
        times = group['ISO_TIME'].values
        for k in range(1, len(idxs)):
            try:
                dist_km = haversine_km(lats[k-1], lons[k-1], lats[k], lons[k])
                dt_hr   = (times[k] - times[k-1]) / np.timedelta64(1, 'h')
                if dt_hr > 0:
                    speeds[idxs[k]] = dist_km / dt_hr
            except Exception:
                pass
    df['TRACK_SPEED'] = speeds
    return df


def create_labels(df, start_hours, end_hours, threshold=15):
    df = df.copy().sort_values(['SID', 'ISO_TIME']).reset_index(drop=True)
    targets = np.full(len(df), np.nan)
    for sid, group in df.groupby('SID', sort=False):
        idxs  = group.index.tolist()
        times = group['ISO_TIME'].values
        winds = group['WIND'].values
        for pos, i in enumerate(idxs):
            t = times[pos]
            w_curr = winds[pos]
            dt_hours = (times - t) / np.timedelta64(1, 'h')
            mask = (dt_hours >= start_hours) & (dt_hours <= end_hours)
            future_winds = winds[mask]
            if len(future_winds) == 0:
                targets[i] = np.nan
            else:
                max_future = float(np.nanmax(future_winds))
                targets[i] = 1.0 if (max_future - w_curr) >= threshold else 0.0
    df['TARGET'] = targets
    return df


df = add_track_features(df)
df_short = create_labels(df, start_hours=48,  end_hours=72)
df_long  = create_labels(df, start_hours=120, end_hours=168)

print('Short horizon target dist:')
print(df_short['TARGET'].value_counts(dropna=False))
print('\nLong horizon target dist:')
print(df_long['TARGET'].value_counts(dropna=False))

In [ ]:
# ── Training & Evaluation with Weather Features ───────────────────────────────
BASE_FEATURES    = ['LAT', 'LON', 'WIND', 'MONTH', 'HOUR',
                    'LAT_CHANGE', 'LON_CHANGE', 'WIND_CHANGE', 'TRACK_SPEED']
WEATHER_FEATURES = ['SST', 'MSL', 'U10', 'V10', 'T2M', 'D2M', 'TP']


def train_and_evaluate(df_labeled, horizon_label=''):
    weather_available = [c for c in WEATHER_FEATURES if c in df_labeled.columns]
    feat_cols = BASE_FEATURES + weather_available
    feat_cols = [c for c in feat_cols if c in df_labeled.columns]

    df_clean = df_labeled.dropna(subset=['TARGET'])
    X = df_clean[feat_cols].values
    y = df_clean['TARGET'].astype(int).values
    groups = df_clean['SID'].values

    if len(np.unique(groups)) < 2:
        print(f'[{horizon_label}] Not enough storms — skipping.')
        return None, feat_cols

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups))

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    print(f'\n{"=" * 60}')
    print(f'  {horizon_label}')
    print(f'  Features used ({len(feat_cols)}): {feat_cols}')
    print(f'  Train: {len(y_train)} | Test: {len(y_test)}')

    models = {
        'RandomForest': Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('classifier', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
        ]),
        'XGBoost': Pipeline([
            ('imputer',    SimpleImputer(strategy='median')),
            ('classifier', XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                         use_label_encoder=False, eval_metric='logloss',
                                         random_state=42, verbosity=0)),
        ]),
    }

    xgb_pipe = None
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)
        y_prob = pipe.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)
        print(f'\n  ── {name} ──')
        print(classification_report(y_test, y_pred, zero_division=0))
        print('  Confusion matrix:')
        print(confusion_matrix(y_test, y_pred))
        if len(np.unique(y_test)) >= 2:
            print(f'  ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}')
            print(f'  PR-AUC  : {average_precision_score(y_test, y_prob):.4f}')
        else:
            print('  [WARNING] Only one class in test set — skipping ROC/PR-AUC')
        print(f'  Brier   : {brier_score_loss(y_test, y_prob):.4f}')
        if name == 'XGBoost':
            xgb_pipe = pipe

    return xgb_pipe, feat_cols


xgb_short, feat_short = train_and_evaluate(df_short, 'SHORT HORIZON (48–72 h)')
xgb_long,  feat_long  = train_and_evaluate(df_long,  'LONG HORIZON (120–168 h)')

In [ ]:
# ── V1 vs V2 Feature Comparison ───────────────────────────────────────────────
v1_features = ['LAT', 'LON', 'WIND', 'MONTH', 'HOUR',
               'LAT_CHANGE', 'LON_CHANGE', 'WIND_CHANGE', 'TRACK_SPEED']
v2_new      = [c for c in (WEATHER_FEATURES) if c in df_short.columns]

print('V1 features:', v1_features)
print('V2 new weather features available:', v2_new)
print(f'Total V2 features: {len(v1_features) + len(v2_new)}')

In [ ]:
# ── Example Prediction ────────────────────────────────────────────────────────
example_obs = {
    'LAT': 15.0,  'LON': 88.5,  'WIND': 45.0,
    'MONTH': 10.0, 'HOUR': 6.0,
    'LAT_CHANGE': 0.3,  'LON_CHANGE': -0.2,
    'WIND_CHANGE': 5.0, 'TRACK_SPEED': 12.0,
    # ERA5 values (replace with real values from your dataset)
    'SST': 302.0,  # K
    'MSL': 100200.0,  # Pa
    'U10': -5.0,   # m/s
    'V10':  2.0,   # m/s
    'T2M': 301.0,  # K
    'D2M': 299.0,  # K
    'TP':  0.005,  # m
}

if xgb_short is not None:
    X_new = pd.DataFrame([example_obs])[feat_short].values
    prob  = xgb_short.predict_proba(X_new)[0, 1]
    print(f'Short-horizon (48–72 h) intensification probability : {prob:.3f}')

if xgb_long is not None:
    X_new = pd.DataFrame([example_obs])[feat_long].values
    prob  = xgb_long.predict_proba(X_new)[0, 1]
    print(f'Long-horizon  (120–168 h) intensification probability: {prob:.3f}')